<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [1]:
#%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

In [2]:
# Params
VERBOSE = True
CHOSEN = 'llama'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
IT = 10
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'
OVERWRITE = ['']
MAX_RETRY = 10
try:
    #DRIVE
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    BASE_FOLDER = 'drive/MyDrive/NLP_proj/estimation/'
    N_GPU_LAYERS = -1
except:
    #LOCAL
    BASE_FOLDER = 'estimation/'
    N_GPU_LAYERS = 20

In [3]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os



# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q6_K.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"bartowski/Qwen2.5-7B-Instruct-GGUF",
             'filename': "Qwen2.5-7B-Instruct-Q6_K.gguf",
             'temperature': 0.6,
             'n_ctx': 40960,
             'chat_format': "qwen"},
    'gemma': {'repo_id':"bartowski/google_gemma-3n-E4B-it-GGUF",
                'filename':"google_gemma-3n-E4B-it-Q6_K.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': None
             },
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=N_GPU_LAYERS, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=VERBOSE,
                            force_download=True,
                            enable_thinking=True)


def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]



prompts = {}
rfs={}

/home/prometheus/anaconda3/envs/llama-env/lib/python3.9/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce GTX 1070, compute capability 6.1, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce GTX 1070) - 7708 MiB free
llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from /home/prometheus/.cache/huggingface/hub/models--bartowski--Meta-Llama-3.1-8B-Instruct-GGUF/snapshots/bf5b95e96dac0462e2a09145ec66cae9a3f12067/./Meta-Llama-3.1-8B-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:  

In [4]:
import re
test1 = 'mechanics: [A, B, ..., Z]'
test2 = 'a\nbhst\nOverall Complexity: 4.2'
test3 ='tsr\n*optimal number of pLayers: 10*'
test4 ='tsr\noptimal Player count: 1-2'
test_dur_1 = 'sthser\nduration: 10-20'
test_dur_2 = 'sthser\nduration: 120-30 minutes'

regex = {'mechanics': r"(?i)^.*\bmechanics\s*:\s*\[.*\]\s*$",
         'complexity': r'(?i)^.*\bcomplexity:\s*[1-5]\.\d\s*$',
         'player': r'(?i)^.*\bplayer.*:\s*\d+',
         'duration': r'(?i)^.*\bduration.*:\s*(\d+)(?:-(\d+))?'

}
def check_output(prompt, output):
    if(prompt == 'all'):
        try:
            out_dic = json.loads(output)
            if('answer' in out_dic):
                return True
            else:
                return False
        except:
            return False
    else:
        last_line = output.split('\n')[-1]
        return bool(re.match(regex[prompt], last_line))


print(check_output('mechanics',test1))
print(check_output('complexity',test2))
print(check_output('player',test3))
print(check_output('player',test4))
print(check_output('duration',test_dur_1))
print(check_output('duration',test_dur_2))


True
True
True
True
True
True


# Game parameter estimation
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

## Estimating everything at once

In [5]:

prompts['all'] = """You are a board‑game analyst that always explains its reasoning before answering.
For any supplied rule excerpt you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting already existing BGG mechanic(s).
3. Judge the rule density and decision depth. Then assign a complexity score (1.0‑5.0).
4. From the number of components in the box and player‑interaction patterns infer the optimal player‑count.
6. Assess the progression of a typical game turn. Based on the complexity of the required actions and how much each turn brings the player closer to the final goal, estimate the game’s average duration in minutes.
Explain your reasoning step by step then output a json object with the fields 'mechanics' (list of strings), 'complexity'(1-5), 'optimal player count', 'duration' that matches the schema below.

<<<JSON schema>>>
{
  "type": "object",
  "properties": {
    "reasoning": {"type": "string"},
    "answer": {"type": "object", "properties": {
      "mechanics": {"type": "array", "items": {"type": "string"}},
      "complexity": {"type": "number", "minimum": 1.0, "maximum": 5.0},
      "optimal player count": {"type": "number"},
      "duration": {"type": "number"},
      "required": ["mechanics", "complexity", "optimal player count", "duration"]
    }}
  },
  "required": ["reasoning", "answer"]
}


<<<EXAMPLE>>>
{
  "reasoning": "The rules describe moving pieces on a grid, controlling regions, and a simple scoring system. These map to Area Control and Hand Management. The rule density is low and decisions are straightforward, so complexity is around 1.8. With only 30 tokens and a small board, the game works best with 2‑3 players; 2 is optimal for maximum interaction. Each turn shifts control of a few squares, and a full game finishes in roughly 30 minutes.",
  "answer": {
    "mechanics": ["Area Control", "Hand Management"],
    "complexity": 1.8,
    "optimal player count": 2,
    "duration": 30
  }
}
"""

rfs['all'] = {"type": "json_object",
          "schema": {
              "type": "object",
              "properties": {
                  "reasoning": {"type": "string"},
                  "answer": {"type": "object", "properties": {
                      "mechanics": {"type": "array", "items": {"type": "string"}},
                      "complexity": {"type": "number", "minimum": 1.0, "maximum": 5.0},
                      "optimal player count": {"type": "number"},
                      "duration": {"type": "number"},
                      "required": ["mechanics", "complexity", "optimal player count", "duration"]
                      }
                    }
                  },
              "required": ["reasoning", "answer"]
              }
          }



## Estimating each parameter separately

### Mechanics

In [6]:
prompts['mechanics'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting BGG mechanic(s). Use only mechanics present in the list below.

here is a complete list of the available BGG mechanics:
[Acting, Action / Event, Action Drafting, Action Points, Action Queue, Action Retrieval,
Action Timer, Advantage Token, Alliances, Area Majority / Influence, Area Movement, Area-Impulse,
Auction / Bidding, Auction Compensation, Auction: Dexterity, Auction: Dutch, Auction: Dutch Priority,
Auction: English, Auction: Fixed Placement, Auction: Multiple Lot, Auction: Once Around,
Auction: Sealed Bid,Auction: Turn Order Until Pass, Automatic Resource Growth, Betting and Bluffing,
Bias, Bids As Wagers, Bingo, Bribery, Campaign / Battle Card Driven, Card Play Conflict Resolution,
Catch the Leader, Chaining, Chit-Pull System, Closed Drafting, Closed Economy Auction, Command Cards,
Commodity Speculation, Communication Limits, Connections, Constrained Bidding, Contracts,
Cooperative Game, Crayon Rail System, Critical Hits and Failures, Cube Tower, Deck Construction,
"Deck, Bag, and Pool Building", Deduction,Delayed Purchase, Dice Rolling, Die Icon Resolution,
Different Dice Movement, Drawing, Elapsed Real Time Ending, Enclosure, End Game Bonuses, Events,
Finale Ending, Flicking, Follow, Force Commitment, Grid Coverage, Grid Movement, Hand Management,
Hexagon Grid, Hidden Movement, Hidden Roles, Hidden Victory Points, Highest-Lowest Scoring, Hot Potato,
"I Cut, You Choose", Impulse Movement, Income, Increase Value of Unchosen Resources, Induction,
Interrupts, Investment, Kill Steal, King of the Hill, Ladder Climbing, Layering,
Legacy Game, Line Drawing, Line of Sight, Loans, Lose a Turn, Mancala,
Map Addition, Map Deformation, Map Reduction, Market, Matching, Measurement Movement,
Melding and Splaying, Memory, Minimap Resolution, Modular Board, Move Through Deck,
Movement Points, Movement Template, Moving Multiple Units, Multi-Use Cards, Multiple Maps,
Narrative Choice / Paragraph, Negotiation, Neighbor Scope, Network and Route Building,
Once-Per-Game Abilities, Open Drafting, Order Counters, Ordering, Ownership, Paper-and-Pencil,
Passed Action Token, Pattern Building, Pattern Movement, Pattern Recognition, Physical Removal,
Pick-up and Deliver, Pieces as Map, Player Elimination, Player Judge, Point to Point Movement,
Predictive Bid, Prisoner's Dilemma, Programmed Movement, Push Your Luck, Questions and Answers, Race,
Random Production, Ratio / Combat Results Table, Re-rolling and Locking, Real-Time, Relative Movement,
Resource Queue, Resource to Move, Rock-Paper-Scissors, Role Playing, Roles with Asymmetric Information,
Roll / Spin and Move, Rondel, Scenario / Mission / Campaign Game, Score-and-Reset Game, Secret Unit Deployment,
Selection Order Bid, Semi-Cooperative Game, Set Collection, Simulation, Simultaneous Action Selection,
Singing, Single Loser Game, Slide / Push, Solo / Solitaire Game, Speed Matching, Spelling, Square Grid,
Stacking and Balancing, Stat Check Resolution, Static Capture, Stock Holding, Storytelling, Sudden Death Ending,
Tags, Take That, Targeted Clues, Team-Based Game, Tech Trees / Tech Tracks, Three Dimensional Movement,
Tile Placement, Track Movement, Trading, Traitor Game, Trick-taking, Tug of War, Turn Order: Auction,
Turn Order: Claim Action, Turn Order: Pass Order, Turn Order: Progressive, Turn Order: Random,
Turn Order: Role Order, Turn Order: Stat-Based, Turn Order: Time Track, Variable Phase Order,
Variable Player Powers, Variable Set-up, Victory Points as a Resource, Voting, Worker Placement,
Worker Placement with Dice Workers, "Worker Placement, Different Worker Types", Zone of Control]

after your reasoning output one last line with the chosen mechanics.

**Final Output**
mechanics: [A, B, ..., Z]
"""
rfs['mechanics'] = {"type": "json_object",
                   "schema": {
                       "type": "object",
                       "properties": {
                           "reasoning": {"type": "string"},
                           "answer": {"type": "array", "items": {"type": "string"}}
                       },
                       "required": ["reasoning", "answer"]
                   }
                  }


### Complexity rating

In [7]:
prompts['complexity'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Analyze Learning Complexity
- Analyze the length of the text, setup steps, rule exceptions and other factor that may indicate how rule intensive the game is.
- Reason about how quickly a new player could grasp the basics.
2. Analyze Playing Complexity
- Look at in‑game actions per turn, resource management, simultaneous moves and number of element to manage
- Estimate mental load during a typical play session.
3. Analyze Strategy/Tactics
- Examine depth of decision space, long‑term planning, branching possibilities, and how impactful is a wrong decision.
4. Convert each qualitative assessment to a numeric rating (1‑5)
- Provide a short justification for each number.
5 Compute the complexity rating
- Average = (Learning + Playing + Strategy) / 3
- Round to one decimal.

After your reasoning output one last line with the overall complexity rating.

**Final Output**
Overall Complexity: W.W"""

rfs['mechanics'] = {"type": "json_object",
                   "schema": {
                       "type": "object",
                       "properties": {
                           "reasoning": {"type": "string"},
                           "answer": {"type": "array", "items": {"type": "string"}}
                       },
                       "required": ["reasoning", "answer"]
                   }
                  }

### Optimal player count

In [8]:
prompts['player'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Identify the player‑count range stated in the rules (minimum‑maximum). If the rulebook does not explicitly state a player count, infer the appropriate range from the game components and mechanics described in the box contents.
2. Examine how the core mechanics scale with player number
3. Consider the impact on play time, player interaction, and variance (e.g., games that become chaotic with many players or too slow with few).
4. Weigh the pros and cons of each possible player count within the allowed range.
5. find the optimal player count (a single number not a range) based on your observations

After your reasoning output one last line with the optimal player count.

**Final Output**
Optimal player count: X
"""

### Game duration

In [9]:
prompts['duration'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Assess the progression of a typical game turn.
2. Evaluate the complexity of the required actions and how long a turn would last
3. Evaluate how much each turn brings the player closer to the final goal
4. Based on your observation estimate the game’s average duration in minutes.

After your reasoning output one final line wit the expected game duration.

**Final Output**
Average game duration: X minutes
"""

## Execution

In [10]:
# select only prompt not executed
to_do = [g for g in FILE_NAMES if f'{CHOSEN}_{g}.json' not in os.listdir(BASE_FOLDER)
                               or f'{CHOSEN}_{g}.json' in OVERWRITE]
if to_do == []:
    print('Nothing to do')

for g in to_do:
    print(g)
    output_dict = {p: {it: '' for it in range(IT)} for p in prompts.keys()}
    for p,it in tqdm([(p,it) for p in reversed(list(prompts.keys())) for it in range(IT)]):
        for retry in range(MAX_RETRY):
            if(retry > 0 and VERBOSE):
                print(f'Retry {retry}')
            rulebook = requests.get(BASE_URL +g+'.txt').text
            name = g.replace('_',' ')
            out = model.create_chat_completion(generate_message(prompts[p], f'Here is the full rulebook of the game {name}:\n'+rulebook),
                                               temperature=models[CHOSEN]['temperature'],
                                               response_format = rfs['all'] if p == 'all' else None,
                                               )['choices'][0]['message']['content']
            if(check_output(p,out)):
                break
        output_dict[p][it] = out
        if(VERBOSE):
            print(p,'-',it)
            print(out)

    with open(f'{BASE_FOLDER}{CHOSEN}_{g}.json','w') as f:
        json.dump(dict(output_dict),f)

power_grid_recharged


  0%|                                                    | 0/50 [00:00<?, ?it/s]llama_perf_context_print:        load time =   25253.21 ms
llama_perf_context_print: prompt eval time =   25250.85 ms /  7498 tokens (    3.37 ms per token,   296.94 tokens per second)
llama_perf_context_print:        eval time =  180365.61 ms /   418 runs   (  431.50 ms per token,     2.32 tokens per second)
llama_perf_context_print:       total time =  206114.60 ms /  7916 tokens
llama_perf_context_print:    graphs reused =        416
  2%|▊                                        | 1/50 [03:26<2:48:38, 206.50s/it]Llama.generate: 7497 prefix-match hit, remaining 1 prompt tokens to eval


duration - 0
After carefully reading the rulebook, I will provide my analysis of the game Power Grid Recharged.

**Assessment of the progression of a typical game turn:**

A typical game turn consists of five phases: Determine Player Order, Auction Power Plants, Buy Resources, Build Houses, and Bureaucracy. Each phase has specific rules and actions that players must take in a predetermined order. The game is divided into three Steps: Step 1, Step 2, and Step 3, each with its own set of rules and changes.

**Evaluation of the complexity of the required actions and how long a turn would last:**

The game requires strategic planning and management of resources, power plants, and city connections. Each phase has its own complexity level, with the Auction Power Plants phase being the most complex due to the bidding and auction mechanics. The Buy Resources phase is also complex due to the need to manage resources and power plant storage. The Build Houses phase requires strategic planning to 

llama_perf_context_print:        load time =   25253.21 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  197333.37 ms /   459 runs   (  429.92 ms per token,     2.33 tokens per second)
llama_perf_context_print:       total time =  197889.75 ms /   460 tokens
llama_perf_context_print:    graphs reused =        456


Retry 1


...

all - 7
{ "reasoning": "The rules of Power Grid Recharged describe a game of strategic planning and resource management. Key actions include determining player order, auctioning power plants, buying resources, building houses, and producing electricity to supply networks. The game also involves resupplying the resource market and updating the power plant market. Components include power plants, resources, houses, and a scoring track. The game has three Steps, each with unique rules and changes to the game state. The game ends when a player connects the required number of cities, and the winner is the player who can power the most cities in their network using the resources and power plants they have. This maps to the following BGG mechanics: Area Control, Resource Management, Auction/Bidding, and Worker Placement. The rule density is moderate, with many specific rules and exceptions, but the decisions are not overly complex. I would rate the complexity as 3.5 out of 5. Based on the num

llama_perf_context_print:        load time =   25253.21 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   86240.30 ms /   196 runs   (  440.00 ms per token,     2.27 tokens per second)
llama_perf_context_print:       total time =   90466.64 ms /   197 tokens
llama_perf_context_print:    graphs reused =        194
 98%|███████████████████████████████████████▏| 49/50 [3:53:20<02:14, 134.11s/it]Llama.generate: 7886 prefix-match hit, remaining 1 prompt tokens to eval


all - 8
{ "reasoning": "The rules describe a complex game with multiple mechanics, including resource management, auctioning, and city building. These map to Auction/Bidding, Resource Management, and Area Control. The rule density is medium to high, with many specific rules for each phase and step of the game. Decisions are complex, with many variables to consider, so complexity is around 4.2. With 22 houses and 50 Elektro for each player, the game works best with 2-3 players. Each turn involves auctioning, resource management, and city building, and a full game can take several hours to finish, but the actual duration will depend on the number of players and their level of experience.", "answer": { "mechanics": ["Auction/Bidding", "Resource Management", "Area Control"], "complexity": 4.2, "optimal player count": 2, "duration": 180 } }


llama_perf_context_print:        load time =   25253.21 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  100417.88 ms /   228 runs   (  440.43 ms per token,     2.27 tokens per second)
llama_perf_context_print:       total time =  105394.28 ms /   229 tokens
llama_perf_context_print:    graphs reused =        226
100%|████████████████████████████████████████| 50/50 [3:55:05<00:00, 282.12s/it]

all - 9
{ "reasoning": "The rules describe a complex game of resource management, trading, and network building. The game has multiple phases and steps, with each phase and step introducing new mechanics and challenges. The players must manage their resources, build their network, and make strategic decisions to earn the most money. The game has a high level of complexity, with many interacting mechanics and a large number of components. The optimal player count is 2-5, with 3-4 being the most common and optimal. The game's average duration is around 60-90 minutes, with each turn bringing the players closer to the final goal of building the most valuable network. The game's complexity and length make it a challenging and engaging experience for players." , "answer": { "mechanics": ["Resource Management", "Trading", "Network Building", "Auctioning", "Player Order", "Resupply", "Power Plants", "Cities", "Electricty", "Resource Tokens"], "complexity": 4.5, "optimal player count": 3, "dura